# Etapa 4: Modelado — Predicción de Churn

Este notebook corresponde a la Etapa 4 del ciclo CRISP-DM del proyecto. Parte del dataset
ya limpio y transformado en la Etapa 3 (`data/processed/X_features.csv` y `y_target.csv`)
para entrenar y comparar tres algoritmos de clasificación: Regresión Logística, Random
Forest y Gradient Boosting.

El objetivo no es maximizar una métrica en abstracto, sino identificar qué modelo detecta
mejor a los clientes en riesgo real de fuga (Recall, Precision, F1), evaluado con
Stratified K-Fold Cross-Validation para obtener una estimación robusta del desempeño.
El modelo (o modelos) seleccionados acá van a ser la base de la Etapa 5, donde se traduce
su desempeño en impacto económico real para el negocio.

**Nota**: el ajuste del umbral de decisión (más allá del 0.5 por default) se deja para la
Etapa 5, una vez definida la lógica de costos y beneficios.

In [1]:
import pandas as pd

X = pd.read_csv('../data/processed/X_features.csv')
y = pd.read_csv('../data/processed/y_target.csv')

In [2]:
print(X.shape)
print(y.shape)

(7043, 29)
(7043, 1)


## Train/Test Split

Se separa el dataset completo en dos partes: un 80% para desarrollo (entrenamiento y
cross-validation de los distintos modelos) y un 20% que queda completamente aislado
como "examen final", sin participar en ninguna decisión durante el desarrollo del modelo.

La división usa `stratify=y` para preservar la proporción real de churn (73.46%/26.54%)
tanto en train como en test, evitando que la partición al azar genere un desbalance
distinto entre ambos grupos. `random_state=42` fija la aleatoriedad para que la división
sea siempre la misma cada vez que se corra el notebook, garantizando reproducibilidad.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [4]:
print(X_train.shape)
print(X_test.shape)
print(y_train['Churn Value'].mean())
print(y_test['Churn Value'].mean())

(5634, 29)
(1409, 29)
0.2653532126375577
0.2654364797728886


### Resultado: Train/Test Split

Se confirma la división 80/20 (5,634 filas train / 1,409 filas test), con la proporción
de churn prácticamente idéntica en ambos grupos (26.54% train, 26.54% test) — el
parámetro `stratify=y` cumplió su función correctamente. El set de test queda apartado
y no se vuelve a tocar hasta la evaluación final del modelo elegido.

## Configurar Stratified K-Fold

Se configura el objeto Stratified K-Fold que se va a usar para evaluar cada modelo
durante el desarrollo, dividiendo el set de entrenamiento (X_train, y_train) en 5
partes que mantienen la proporción real de churn (73%/27%) en cada una. A diferencia
del Train/Test Split del paso anterior, acá no se dividen los datos todavía — se define
la configuración que se va a reutilizar repetidamente en el paso 4, al entrenar y
comparar los tres modelos.

In [5]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Resultado: Stratified K-Fold configurado

Se configuró `skf` con 5 folds, mezcla aleatoria y semilla fija (random_state=42).
Este objeto no divide los datos por sí mismo — define la estrategia que se va a
aplicar en el paso 4 al entrenar y comparar los tres modelos.

## Entrenar y comparar los 3 modelos

Se entrenan Regresión Logística, Random Forest y Gradient Boosting usando el mismo
Stratified K-Fold (5 folds), registrando las mismas métricas para los tres en igualdad
de condiciones (Recall, Precision, F1, ROC-AUC de la clase Churn=1). Como es el mismo
proceso repetido para cada modelo, se escribe una única función reutilizable en
`src/modeling.py`, siguiendo el principio DRY (Don't Repeat Yourself).

### Primero cross_validate con un solo modelo (Regresión Logística)

Antes de armar la función reutilizable para los 3 modelos, se prueba `cross_validate`
con un solo modelo (Regresión Logística) para entender su funcionamiento: en cada una
de las 5 rondas del Stratified K-Fold, entrena un modelo nuevo con 4 folds y lo evalúa
con el fold restante, registrando Recall, Precision, F1 y ROC-AUC de la clase Churn=1
en cada ronda.

La primera corrida arrojó un `ConvergenceWarning` repetido en las 5 rondas: el modelo
no logró estabilizarse dentro de las 1000 iteraciones permitidas. La causa es que las
variables numéricas del dataset están en escalas muy distintas entre sí (columnas
binarias de 0-1 conviven con `Total Charges`, que llega a ~8,000), algo ya anticipado
en el planning de la Etapa 3.

La corrección usa un `Pipeline` que encadena `StandardScaler` y el modelo como un solo
objeto — así, el escalado se recalcula dentro de cada fold usando solo sus datos de
entrenamiento, evitando que información del fold de evaluación se filtre en el cálculo
del escalado (data leakage).

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

modelo_escalado = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])

metricas = ['recall', 'precision', 'f1', 'roc_auc']

resultados = cross_validate(
    modelo_escalado, 
    X_train, 
    y_train['Churn Value'], 
    cv=skf, 
    scoring=metricas
)

resultados

{'fit_time': array([0.02377748, 0.02264071, 0.01939297, 0.02499509, 0.02025843]),
 'score_time': array([0.01131606, 0.0101521 , 0.01028776, 0.01005411, 0.01093102]),
 'test_recall': array([0.83946488, 0.77257525, 0.78929766, 0.85618729, 0.7993311 ]),
 'test_precision': array([0.5504386 , 0.51910112, 0.50752688, 0.53222453, 0.5456621 ]),
 'test_f1': array([0.66490066, 0.62096774, 0.61780105, 0.65641026, 0.64857531]),
 'test_roc_auc': array([0.86212092, 0.837932  , 0.85335175, 0.87106983, 0.87069757])}

#### Promedio de las métricas de Regresión Logística

Se calcula el promedio de las 5 rondas para cada métrica, resumiendo el desempeño
general del modelo en un solo número por métrica (en vez de tener que leer los 5
valores sueltos cada vez).

In [7]:
for metrica in metricas:
    promedio = resultados[f'test_{metrica}'].mean()
    print(f"{metrica}: {promedio:.4f}")

recall: 0.8114
precision: 0.5310
f1: 0.6417
roc_auc: 0.8590


##### Resultado: Regresión Logística (con class_weight='balanced')

Recall: 0.8114 | Precision: 0.5310 | F1: 0.6417 | ROC-AUC: 0.8590

El modelo detecta correctamente el 81% de los clientes que realmente se van a ir, a
costa de una Precision más baja (53%) — consistente con `class_weight='balanced'`,
que prioriza detectar la clase minoritaria aunque genere más falsos positivos. El
ROC-AUC de 0.86 indica una buena capacidad general de separación entre ambas clases.

### Random Forest

Se evalúa Random Forest con el mismo Stratified K-Fold y las mismas métricas que
Regresión Logística, reutilizando la función `evaluar_modelo()` de `src/modeling.py`.
A diferencia de Regresión Logística, no necesita escalado de variables (no usa Pipeline),
ya que los modelos basados en árboles no son sensibles a la escala de los datos.

In [8]:
from sklearn.ensemble import RandomForestClassifier

modelo_rf = RandomForestClassifier(class_weight='balanced', random_state=42)

In [10]:
import sys
sys.path.append('..')

from src.modeling import evaluar_modelo

resultados_rf = evaluar_modelo(modelo_rf, X_train, y_train['Churn Value'], skf, metricas)

Resultados promedio:
recall: 0.6689
precision: 0.5857
f1: 0.6245
roc_auc: 0.8430


#### Resultado: Random Forest (con class_weight='balanced')

Recall: 0.6689 | Precision: 0.5857 | F1: 0.6245 | ROC-AUC: 0.8430

Comparado con Regresión Logística, Random Forest muestra menor Recall (67% vs 81%) pero
mayor Precision (59% vs 53%) — detecta menos casos reales de churn, pero se equivoca
menos cuando marca a alguien como riesgo. El F1 y ROC-AUC quedan parejos entre ambos
modelos, sin un ganador claro todavía.

### Gradient Boosting

Se evalúa Gradient Boosting con el mismo Stratified K-Fold y las mismas métricas que
los dos modelos anteriores, reutilizando `evaluar_modelo()`. Al igual que Random Forest,
no necesita escalado de variables por ser un modelo basado en árboles.

In [11]:
from sklearn.ensemble import GradientBoostingClassifier

modelo_gb = GradientBoostingClassifier(random_state=42)

In [12]:
resultados_gb = evaluar_modelo(modelo_gb, X_train, y_train['Churn Value'], skf, metricas)

Resultados promedio:
recall: 0.5472
precision: 0.6593
f1: 0.5977
roc_auc: 0.8625


#### Resultado: Gradient Boosting (sin ajuste de balance)

Recall: 0.5472 | Precision: 0.6593 | F1: 0.5977 | ROC-AUC: 0.8625

Es el modelo más conservador de los tres: menor Recall pero mayor Precision. Sin
embargo, tiene el ROC-AUC más alto (86.25%), indicando la mejor capacidad de separación
general entre clases — una señal de que el ajuste del umbral de decisión en la Etapa 5
podría mejorar significativamente su Recall sin perder esa buena separación de base.

## Comparación de los 3 modelos

Se consolidan los resultados promedio de Regresión Logística, Random Forest y Gradient
Boosting en una sola tabla, para facilitar la comparación directa antes de decidir cuál
(o cuáles) avanzan a la Etapa 5.

In [13]:
comparacion_resultados = pd.DataFrame({
    'Logistic Regression': [resultados[f'test_{metrica}'].mean() for metrica in metricas],
    'Random Forest': [resultados_rf[f'test_{metrica}'].mean() for metrica in metricas],
    'Gradient Boosting': [resultados_gb[f'test_{metrica}'].mean() for metrica in metricas]
}, index=metricas)

comparacion_resultados

,Logistic Regression,Random Forest,Gradient Boosting
recall,0.811371,0.668896,0.547157
precision,0.530991,0.585675,0.659305
f1,0.641731,0.624479,0.597667
roc_auc,0.859034,0.842971,0.862539


### Resultado: tabla comparativa

|              | Regresión Logística | Random Forest | Gradient Boosting |
|--------------|---------------------|----------------|---------------------|
| Recall       | 0.8114              | 0.6689         | 0.5472               |
| Precision    | 0.5310              | 0.5857         | 0.6593               |
| F1           | 0.6417              | 0.6245         | 0.5977               |
| ROC-AUC      | 0.8590              | 0.8430         | 0.8625               |

No hay un ganador absoluto: Regresión Logística domina en Recall, Gradient Boosting en
Precision y ROC-AUC, y los tres quedan relativamente parejos en F1. La decisión de qué
modelo(s) avanzan a la Etapa 5 depende del contexto de negocio (impacto económico de
Falsos Positivos vs. Falsos Negativos), no solo de estas métricas en abstracto.

## Gráfico: comparación visual de los 3 modelos

Se grafica la tabla comparativa como barras agrupadas, para visualizar de forma más
directa los trade-offs entre modelos (Recall vs. Precision, especialmente).

In [16]:
comparacion_larga = comparacion_resultados.reset_index().melt(id_vars='index', var_name='Modelo', value_name='Valor')
comparacion_larga.columns = ['Métrica', 'Modelo', 'Valor']
comparacion_larga

,Métrica,Modelo,Valor
0,recall,Logistic Regression,0.811371
1,precision,Logistic Regression,0.530991
2,f1,Logistic Regression,0.641731
3,roc_auc,Logistic Regression,0.859034
4,recall,Random Forest,0.668896
5,precision,Random Forest,0.585675
6,f1,Random Forest,0.624479
7,roc_auc,Random Forest,0.842971
8,recall,Gradient Boosting,0.547157
9,precision,Gradient Boosting,0.659305


### Resultado: transformación a formato largo

Se convirtió la tabla comparativa de formato ancho (4 filas × 3 columnas) a formato
largo (12 filas × 3 columnas) usando `.melt()`, donde cada fila representa una
combinación única de métrica + modelo + valor — el formato que requiere Plotly para
graficar barras agrupadas.

In [18]:
from plotly import express as px

fig = px.bar(
    comparacion_larga,
    x='Métrica',
    y='Valor',
    color='Modelo',
    barmode='group',
    title='Comparación de Modelos por Métrica',
    text_auto='.2f'
)
fig.show()

## Selección de modelos finalistas

Se seleccionan dos modelos para continuar a la Etapa 4.7 (feature importance) y a la
Etapa 5 (impacto económico): **Regresión Logística** (mejor Recall, 81%) y **Gradient
Boosting** (mejor ROC-AUC y Precision, con potencial de mejora en Recall al ajustar el
umbral de decisión). Random Forest queda descartado por no destacarse en ninguna métrica
frente a los otros dos.

La decisión final entre los dos finalistas se toma en la Etapa 5, comparando el ahorro
neto económico de cada uno con su umbral óptimo — un criterio de negocio real, en vez
de depender solo de métricas técnicas en abstracto.

## Paso 4.7: Feature Importance de Gradient Boosting

Se entrena Gradient Boosting una vez sobre todo X_train (no en folds, como en
cross-validation) para poder extraer e inspeccionar `feature_importances_` — el
puntaje que el modelo le asigna a cada variable según cuánto la usó para tomar
decisiones. Esto permite confirmar si las variables que ya identificamos como fuertes
en Excel y en los gráficos (Contract, tenure, Payment Method) también son las más
relevantes según el modelo.

In [20]:
modelo_gb.fit(X_train, y_train['Churn Value'])

importancias = pd.DataFrame({
    'Característica': X_train.columns,
    'Importancia': modelo_gb.feature_importances_
}).sort_values(by='Importancia', ascending=False)

importancias

,Característica,Importancia
12,Contract,0.387123
4,Tenure Months,0.142565
20,Internet Service_Fiber optic,0.102761
3,Dependents,0.092289
24,Payment Method_Electronic check,0.050669
15,Total Charges,0.048618
14,Monthly Charges,0.035624
27,Cargo_Promedio_Mensual,0.032821
21,Internet Service_No,0.027559
13,Paperless Billing,0.016229


## Paso 4.7 (continuación): Coeficientes de Regresión Logística

A diferencia de `feature_importances_` (que solo da magnitud), los coeficientes de
Regresión Logística indican también la **dirección** del efecto: un coeficiente
positivo significa que a mayor valor de esa variable, mayor probabilidad de churn;
uno negativo, el efecto contrario. Esto permite entender no solo qué variables pesan,
sino cómo empujan la predicción en cada sentido.

In [24]:
modelo_escalado.fit(X_train, y_train['Churn Value'])

coeficientes = pd.DataFrame({
    'Variable': X_train.columns,
    'Coeficiente': modelo_escalado.named_steps['modelo'].coef_[0]
}).sort_values('Coeficiente', ascending=False)

coeficientes

,Variable,Coeficiente
28,Grupo_Antiguedad,0.702877
20,Internet Service_Fiber optic,0.580710
15,Total Charges,0.455607
11,Streaming Movies,0.215214
10,Streaming TV,0.208645
13,Paperless Billing,0.158520
2,Partner,0.142988
27,Cargo_Promedio_Mensual,0.130785
24,Payment Method_Electronic check,0.127701
18,Multiple Lines_Yes,0.100673
